# Covenant monitoring

**Maintenance** covenants are tested on a schedule (often quarterly); **incurrence** tests gate specific actions. **Springing** covenants activate when a revolver is drawn. Here we **project leverage** and compare it to a threshold under scenarios.

## Concept

The statement model computes **Net Debt / EBITDA** as a tested metric. `covenants.evaluate_engine` applies the covenant package to base and downside snapshots and returns Rust-owned pass/fail and headroom results, including support for richer cure, basket, and step-down policies.

In [1]:
import json

import sys
sys.path.insert(0, "../..")

from _shared import series
from finstack_quant.covenants import cov_lite, evaluate_engine, validate_covenant_engine
from finstack_quant.statements import Evaluator, ModelBuilder
from finstack_quant.statements_analytics import DependencyTracer

PERIODS = ["2025Q1", "2025Q2", "2025Q3", "2025Q4", "2026Q1", "2026Q2"]
AS_OF_DATES = ["2025-03-31", "2025-06-30", "2025-09-30", "2025-12-31", "2026-03-31", "2026-06-30"]

b = ModelBuilder("covenant-demo")
b.periods("2025Q1..2026Q2", "2025Q2")
b.value("revenue", series(PERIODS, [100.0, 102.0, 105.0, 108.0, 110.0, 112.0]))
b.compute("ebitda", "revenue * 0.2")
b.value("total_debt", series(PERIODS, [95.0, 93.0, 91.0, 89.0, 87.0, 85.0]))
b.value("cash", series(PERIODS, [10.0, 10.5, 11.0, 11.5, 12.0, 12.5]))
b.compute("net_debt", "total_debt - cash")
b.compute("leverage", "if(ebitda > 0, net_debt / ebitda, 999.9)")

spec = b.build()
res = Evaluator().evaluate(spec)
print("Dependency tree for leverage:")
print(DependencyTracer(spec).dependency_tree("leverage"))

specs = json.loads(cov_lite(max_leverage=3.75, max_senior_leverage=3.25))
engine_json = json.dumps({"specs": specs, "breach_history": [], "windows": [], "waivers": []})
validate_covenant_engine(engine_json)

for period, as_of in zip(PERIODS, AS_OF_DATES):
    leverage = res.get("leverage", period)
    metrics = json.dumps({"total_leverage": leverage, "senior_leverage": leverage})
    report = json.loads(evaluate_engine(engine_json, metrics, as_of))
    statuses = {covenant_id: item["passed"] for covenant_id, item in report.items()}
    print(period, "leverage=", round(leverage, 3), "engine_results=", statuses)


Dependency tree for leverage:
leverage (if(ebitda > 0, net_debt / ebitda, 999.9))
ebitda (revenue * 0.2)
revenue
net_debt (total_debt - cash)
total_debt
cash

2025Q1 leverage= 4.25 engine_results= {'max_total_leverage': False, 'max_senior_leverage': False, 'negative': True}
2025Q2 leverage= 4.044 engine_results= {'max_total_leverage': False, 'max_senior_leverage': False, 'negative': True}
2025Q3 leverage= 3.81 engine_results= {'max_total_leverage': False, 'max_senior_leverage': False, 'negative': True}
2025Q4 leverage= 3.588 engine_results= {'max_total_leverage': True, 'max_senior_leverage': False, 'negative': True}
2026Q1 leverage= 3.409 engine_results= {'max_total_leverage': True, 'max_senior_leverage': False, 'negative': True}
2026Q2 leverage= 3.237 engine_results= {'max_total_leverage': True, 'max_senior_leverage': True, 'negative': True}


In [2]:
print("Downside metric snapshots through the same covenant engine")
downside_snapshots = {
    "2026Q1": ("2026-03-31", 4.076),
    "2026Q2": ("2026-06-30", 3.940),
}
for period, (as_of, leverage) in downside_snapshots.items():
    metrics = json.dumps({"total_leverage": leverage, "senior_leverage": leverage})
    report = json.loads(evaluate_engine(engine_json, metrics, as_of))
    print(period, {covenant_id: item["passed"] for covenant_id, item in report.items()})
print("All pass/fail decisions above come from covenants.evaluate_engine.")

Downside metric snapshots through the same covenant engine
2026Q1 {'max_total_leverage': False, 'max_senior_leverage': False, 'negative': True}
2026Q2 {'max_total_leverage': False, 'max_senior_leverage': False, 'negative': True}
All pass/fail decisions above come from covenants.evaluate_engine.


## Covenant engine presets (`finstack_quant.covenants`)

The `covenants` module ships standard packages — `cov_lite`, `lbo_standard`, `project_finance`, and `real_estate`. The model supplies tested metrics; `evaluate_engine(engine, metrics, as_of)` remains the single pass/fail and headroom implementation.

In [3]:
from finstack_quant.covenants import (
    lbo_standard,
    project_finance,
    real_estate,
    validate_covenant_report,
    validate_covenant_spec,
)

presets = {
    "cov_lite": cov_lite(max_leverage=5.0, max_senior_leverage=3.5),
    "lbo_standard": lbo_standard(
        initial_leverage=6.0, interest_coverage=2.0, fixed_charge_coverage=1.1, max_capex=50.0
    ),
    "project_finance": project_finance(
        min_dscr=1.2, distribution_lockup_dscr=1.1, min_liquidity=10.0, max_net_leverage=7.0
    ),
    "real_estate": real_estate(min_dscr=1.25, min_debt_yield=0.08, max_ltv=0.75),
}
for name, preset_json in presets.items():
    print(f"{name}: {len(json.loads(preset_json))} covenants")

validate_covenant_spec(json.dumps(specs[0]))
validate_covenant_report(json.dumps(report[next(iter(report))]))
print("Preset spec and latest engine result validated OK")

cov_lite: 3 covenants
lbo_standard: 4 covenants
project_finance: 4 covenants
real_estate: 3 covenants
Preset spec and latest engine result validated OK


## Takeaways

- Encode tested metrics as statement nodes and covenant definitions as engine specs.
- Route every base and downside compliance decision through `covenants.evaluate_engine`.
- Springing and incurrence mechanics belong in engine conditions, windows, and policy-specific specs.